# Eksperyment: Wielo-sezonowy trening + RF vs HGB vs XGBoost (Sprint 6)

## Cel
Bazowa architektura trenuje TYLKO na roku docelowym (~3500 probek), wiec gradient boosting nie ma
jak rozwinac przewagi. Tu trenujemy na WIELU sezonach (domyslnie 2010-2023, ~72 tys. probek),
walidujemy na 2024, testujemy na calym sezonie 2025 -- i porownujemy Random Forest vs
HistGradientBoosting vs XGBoost. Wlasciwy test hipotezy "wiecej danych => boosting wygrywa".

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

## Uruchomienie (od zera): wczytanie wielu sezonow, cechy, strojenie 3 modeli
UWAGA: dlugi bieg (~60-90 min) -- liczenie cech dla ~45 tys. meczow + trening na ~72 tys. probek.

In [2]:
import main_48_cech_multiseason as m
m.main()

Uruchamiam baseline raz (pobranie funkcji feature-engineering)...


Trening 2010-2023 | walidacja 2024 | test 2025


Warmup cech: 2001-2009


Licze cechy dynamiczne dla 41888 meczow (2010-2025) (rozgrzewka: 25265 meczow)...


Mecze: train=36291 val=2950 test=2647


Probki treningowe (po symetryzacji): 72582



[1/3] Random Forest...


[2/3] HistGradientBoosting...


[3/3] XGBoost...



WIELO-SEZONOWY TRENING (2010-2023, ~72582 probek) | test 2025
model             val_match  test_match    Brier  logloss     ECE
RandomForest         0.6475      0.6494   0.2187   0.6260  0.0163
HistGradBoost        0.6468      0.6460   0.2195   0.6283  0.0224
XGBoost              0.6468      0.6494   0.2179   0.6244  0.0212
----------------------------------------------------------------------------------------
DELTA (HistGradBoost - RF): test_match=-0.0034  Brier=+0.0009
DELTA (XGBoost - RF): test_match=+0.0000  Brier=-0.0008

Najlepsze hiperparametry:
  RandomForest: {'n_estimators': 100, 'min_samples_leaf': 10, 'max_samples': 1.0, 'max_features': 'sqrt', 'max_depth': 10}
  HistGradBoost: {'min_samples_leaf': 50, 'max_leaf_nodes': 63, 'max_iter': 300, 'max_features': 1.0, 'learning_rate': 0.03, 'l2_regularization': 1.0}
  XGBoost: {'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.9}

UWAGA: test = caly sezon 2

## Wnioski
Boosting **nie pobil** Random Forest na accuracy nawet na ~72 tys. probek (XGBoost remis, HGB
minimalnie gorzej) -- wszystkie ~65%. Powtorzono tez na ~130 tys. probek (od 2000) -- ten sam
wynik. 3 algorytmy x zakres danych od 3.5k do 130k probek -> wszystko ~65%. **Sciana jest w
cechach/problemie, nie w algorytmie ani ilosci danych.** Jedyna roznica: XGBoost ma minimalnie
lepsza kalibracje (Brier/log-loss), co rosnie z iloscia danych -- istotne tylko gdyby celem byl
betting, nie accuracy.